# **Note for Colab Users**

# **Do not write directly in this file. Your work may disappear!**

# **Always make a copy before you start.**

How to make a copy

1. Click **File** in the top-left corner.  
> *If you cannot see menus like **File** or **Runtime**, click the “v” mark in the top-right corner to show them.*

2. Choose **Save a copy in Drive**.  

3. Rename the copied file to `YOURNAMEs_FileName.ipynb`.  
> Example: If your name is Olivia → `Olivias_FileName.ipynb`  


---

* Check marks (✅) are not saved. They disappear when you refresh the page with Chrome’s reload button.<br>  
If you stop halfway, add a text cell and write something like `SO FAR DONE`.

---

* In Colab, **previous output results are reset every 30 to 90 minutes**.<br>  
So errors like `~~ is not defined` happen **all the time**.

🔁 What should you do when you see a `~~ is not defined` error?

1. First, check the spelling of the variable name.<br>  
2. If the spelling is correct but the error still appears, **click that cell to select it**.<br>  
3. Click **Runtime** → **Run before** in the top-left menu.<br>  
→ This reruns **all cells up to that point**.  
4. Run that cell again.

If the error still does not go away,<br>  
there may be a basic mistake in your TODO answer in an earlier cell.<br>  
Check whether it is correct.<br>  
Or ask ChatGPT or another coding assistant for help.


# **Preparation**

In this part, we only load the content from the previous Chapter.<br>
Just run the code. You do not need to read it carefully.<br>
Feel free to move on.<br>


In [ ]:
# Download the file
!wget https://raw.githubusercontent.com/HayatoHongo/EveryonesLLM/main/input.txt -O input.txt
# Load the downloaded input.txt file with utf-8.
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()

# A function to display tensors in an easy-to-read format (optional)
import torch
import torch.nn as nn
import torch.nn.functional as F

# Install a library that displays tensors in an easy-to-read format
!pip install git+https://github.com/HayatoHongo/print_formatted_tensor.git
# Import a function that displays PyTorch tensors in an easy-to-read format
from torch_print_tensor import print_formatted_tensor



# **Chapter 15: Train nanoGPT with GPU**



### **Section 1: Improving the `Trainer` Class**


Google Colab provides a free T4 GPU.

In Chapter12, we trained on the CPU. This time, we will train seriously using the T4 GPU.


In [ ]:
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'  # Device to use (GPU or CPU)
print(device_type)

**`Check Point`**  
<label><input type="checkbox">Confirmed that the runtime is set to cuda (GPU)<br></label>  


Let’s prepare the Trainer class we defined in Chapter13.

It will be useful later, so inside the Trainer class, we will create dictionaries that record and keep `steps`, `train_loss`, and `val_loss`.

An instance is created with `instance = Class(arguments)`. Compared with a regular variable, a major benefit of an instance is that it can keep information.


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`()`　`[]`　`append`　`add`　`val_losses`　`steps`　`train_losses`


In [ ]:
import time

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

        ########## NEW ##########
        self.steps = __
        self.train_losses = __
        self.val_losses = __
        ########## NEW ##########

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Calculate loss for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Return to training mode again

        # Calculate and return the average loss for each dataset (train, val)
        return {split: sum(values) / len(values) for split, values in losses.items()}

    def train(self):
        # Run train_step the number of times specified in config.
        for step in range(self.config.total_training_steps):

            # Evaluate every 100 times, or only on the final step.
            if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
                # Exclude step==0 because last_eval_end_time is undefined. Exclude the final step because it may be a partial measurement.
                if step == 0 or step == self.config.total_training_steps - 1:
                  tokens_per_second = None
                else:
                  current_eval_start_time = time.time()
                  evaluation_interval = current_eval_start_time - last_eval_end_time
                  tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
                  tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

                eval_loss = self.evaluate()
                print(f"Step {step}: Train Loss {eval_loss['train']:.4f}, Validation Loss {eval_loss['val']:.4f}")
                print(f"Tokens per second {tokens_per_second}")

                ########## NEW ##########
                self._____.append(step)
                self.__________.______(eval_loss['train'])
                self.________.______(eval_loss['val'])
                ########## NEW ##########

                # Record when this evaluation finished. The time difference to the next evaluation start becomes `evaluation_interval`.
                last_eval_end_time = time.time()

            # One training step (the main process done every time)
            train_loss = self.train_step()

<details>
<summary>Click to show/hide the answer</summary>

```python
import time

class Trainer:
    def __init__(self, model, optimizer, data_loader, config):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config

        ########## NEW ##########
        self.steps = []
        self.train_losses = []
        self.val_losses = []
        ########## NEW ##########

    def train_step(self):
        # Get a training batch.
        input_batch, target_batch = self.data_loader.get_batch('train')
        self.optimizer.zero_grad()

        # Model forward pass and loss calculation
        logits, loss = self.model(input_batch, target_batch)
        loss.backward()  # Backpropagation
        self.optimizer.step()  # Parameter update

        return loss.item() # Return the loss value

    def evaluate(self):
        self.model.eval()  # Switch to evaluation mode
        losses = {"train": [], "val": []} # Calculate loss for both training and validation data
        with torch.no_grad():
            for split in ['train', 'val']:
                for _ in range(self.config.evaluation_loops):
                    input_batch, target_batch = self.data_loader.get_batch(split)
                    _, loss = self.model(input_batch, target_batch)
                    losses[split].append(loss.item())
        self.model.train()  # Return to training mode again

        # Calculate and return the average loss for each dataset (train, val)
        return {split: sum(values) / len(values) for split, values in losses.items()}

    def train(self):
        # Run train_step the number of times specified in config.
        for step in range(self.config.total_training_steps):

            # Evaluate every 100 times, or only on the final step.
            if step % self.config.evaluation_frequency == 0 or step == self.config.total_training_steps - 1:
                # Exclude step==0 because last_eval_end_time is undefined. Exclude the final step because it may be a partial measurement.
                if step == 0 or step == self.config.total_training_steps - 1:
                  tokens_per_second = None
                else:
                  current_eval_start_time = time.time()
                  evaluation_interval = current_eval_start_time - last_eval_end_time
                  tokens_per_evaluation_interval = self.config.batch_size * self.config.input_sequence_length * self.config.evaluation_frequency
                  tokens_per_second = tokens_per_evaluation_interval / evaluation_interval

                eval_loss = self.evaluate()
                print(f"Step {step}: Train Loss {eval_loss['train']:.4f}, Validation Loss {eval_loss['val']:.4f}")
                print(f"Tokens per second {tokens_per_second}")

                ########## NEW ##########
                self.steps.append(step)
                self.train_losses.append(eval_loss['train'])
                self.val_losses.append(eval_loss['val'])
                ########## NEW ##########

                # Record when this evaluation finished. The time difference to the next evaluation start becomes `evaluation_interval`.
                last_eval_end_time = time.time()

            # One training step (the main process done every time)
            train_loss = self.train_step()
```


**Section 1: Improving the Trainer Class** <label><input type="checkbox"> Mark as Done</label>


### **Section 2: loss plateau**

Define the `Dataloader` class and the model class. They are exactly the same as in Chpater12.


In [ ]:
class DataLoader:
    def __init__(self, text, config):
        self.config = config  # Config object
        chars = sorted(list(set(text)))  # Sort the unique characters
        self.ctoi = {char: index for index, char in enumerate(chars)}
        self.itoc = {index: char for index, char in enumerate(chars)}
        self.vocab_size = len(chars)

        # Encode and convert to a tensor.
        # You need `self.` to call methods or arguments outside `__init__`.
        self.data = torch.tensor(self.encode(text), dtype=torch.long)

        # Split into training and validation data.
        # Even if no arguments are specified, self.data is used.
        self.train_data, self.val_data = self.split_data()

    def encode(self, text):
        # Convert a string into a sequence of indices. Use self. to call other methods or arguments.
        return [self.ctoi[c] for c in text]

    def decode(self, indices):
        return ''.join([self.itoc[i] for i in indices])

    def split_data(self):
        split_index = int(0.9 * len(self.data))  # The point where 90% of the data is split off for training.
        return self.data[:split_index], self.data[split_index:]

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        start_indices = torch.randint(len(data) - self.config.input_sequence_length, (self.config.batch_size,)) # Generate starting indices for extraction

        input_sequences = torch.stack([
            data[start_index:start_index + self.config.input_sequence_length]
            for start_index in start_indices
        ])
        target_sequences = torch.stack([
            data[start_index + 1:start_index + self.config.input_sequence_length + 1]
            for start_index in start_indices
        ])
        return input_sequences.to(self.config.device_type), target_sequences.to(self.config.device_type)


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Define an embedding table with vocab_size x embedding_dim
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

    def embed(self, input_indices):
        # Get the embedding vectors that correspond to the input indices
        return self.token_embedding_table.forward(input_indices)

class PositionEmbedding(nn.Module):
    def __init__(self, input_sequence_length = 8, embedding_dim = 8):
        super().__init__()
        # Position embedding layer
        self.position_embedding_layer = nn.Embedding(input_sequence_length, embedding_dim)

    def forward(self, input_indices):
        # Shape of the input tensor input_indices: [batch size, sequence length].
        sequence_length = input_indices.shape[1]

        # Create position indices based on the sequence length (example: [0, 1, 2, ..., sequence_length-1])
        position_indices = torch.arange(sequence_length, device=input_indices.device)

        # Get the embedding vectors for the position indices
        position_embeddings = self.position_embedding_layer.forward(position_indices)

        return position_embeddings

class EmbeddingModule(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Embedding layer for each token
        self.token_embedding_layer = TokenEmbedding(vocab_size = vocab_size, embedding_dim = config.embedding_dim)  # Token embedding layer
        self.position_embedding_layer = PositionEmbedding(input_sequence_length = config.input_sequence_length, embedding_dim = config.embedding_dim)  # Embed position information

    def forward(self, input_indices):
        # Get token embeddings
        token_embeddings = self.token_embedding_layer.embed(input_indices)

        # Get position embeddings
        position_embeddings = self.position_embedding_layer.forward(input_indices)

        # Add token embeddings and position embeddings
        embeddings = position_embeddings + token_embeddings
        return embeddings

class AttentionHead(nn.Module):
    def __init__(self, head_size, config):
        super().__init__()
        self.key_fc= nn.Linear(config.embedding_dim, head_size, bias=False)
        self.query_fc = nn.Linear(config.embedding_dim, head_size, bias=False)
        self.value_fc = nn.Linear(config.embedding_dim, head_size, bias=False)

        # Dropout
        self.dropout = nn.Dropout(config.dropout_rate)
        self.head_size = head_size

    def forward(self, input_tensor):
        B, T, C = input_tensor.shape  # Batch, token length, embedding channels

        Key = self.key_fc.forward(input_tensor)     # (B, T, head_size)
        Query = self.query_fc.forward(input_tensor)   # (B, T, head_size)
        Value = self.value_fc.forward(input_tensor)   # (B, T, head_size)

        # Calculate Attention scores (QK^T) / sqrt(embedding_dim)
        attention_weights_before_mask = Query @ Key.transpose(-2, -1) * self.head_size**(-0.5)

        # Mask applied
        mask = torch.triu(torch.ones(T, T), diagonal=1).to(input_tensor.device)
        masked_attention_weights = attention_weights_before_mask.masked_fill(mask == 1, float('-inf'))

        # Softmax -> dropout -> weighted sum
        attention_weights = F.softmax(masked_attention_weights, dim=-1)
        attention_weights = self.dropout(attention_weights)

        out = attention_weights @ Value  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_attention_heads = config.num_attention_heads
        self.embedding_dim = config.embedding_dim
        self.head_size = int(self.embedding_dim / self.num_attention_heads)

        # Manage multiple heads with ModuleList
        self.attention_heads = nn.ModuleList([
            AttentionHead(self.head_size, config)
            for _ in range(self.num_attention_heads)
        ])

        # Linear layer that mixes the outputs of each head
        self.output_projection = nn.Linear(self.embedding_dim, self.embedding_dim)

        # Dropout for the output
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, input_tensor):
        # Get the output from each head
        # A list of (B, T, head_size) tensors
        head_outputs_list = [head.forward(input_tensor) for head in self.attention_heads]

        # Concatenate all head outputs -> (B, T, embedding_dim)
        concatenated = torch.cat(head_outputs_list, dim=-1)

        # Mix outputs with a linear transformation
        projected = self.output_projection.forward(concatenated)

        # Apply dropout to the final output
        output = self.dropout.forward(projected)

        return output

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim),
            nn.Dropout(config.dropout_rate),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Each LayerNorm keeps its own beta and gamma.
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)

        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)

    def forward(self, input_tensor):
        # The forward method is abbreviated.
        normed_input = self.layer_norm1(input_tensor) # Apply LayerNorm to the input
        attention_output = self.multihead_attention(normed_input) # Apply multi-head attention
        residual_attention = attention_output + input_tensor # Add "before! layernorm1"
        normed_attention = self.layer_norm2(residual_attention) # Apply LayerNorm again to the residual output
        feedforward_output = self.feed_forward(normed_attention) # Apply the feed-forward network
        final_output = feedforward_output + residual_attention # Add "before" layernorm2!

        return final_output

class VocabularyLogits(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        # Layer normalization
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        # Projection to the vocabulary size
        self.vocab_projection = nn.Linear(config.embedding_dim, vocab_size)

    def forward(self, transformer_block_output):
        # Apply Layer normalization to the Transformer block output.
        normalized_output = self.output_norm.forward(transformer_block_output)  # (B, T, C)

        # Convert scores to the vocabulary-size dimension with a linear layer.
        vocab_logits = self.vocab_projection.forward(normalized_output)  # (B, T, V)

        return vocab_logits

class nanoGPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.config = config  # Keep this because it is also used during generation.
        self.embedding = EmbeddingModule(vocab_size, config=config)
        self.blocks = nn.Sequential(*[TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(vocab_size=vocab_size, config=config)
        self.criterion = nn.CrossEntropyLoss()

    # Generate text
    def generate(self, input_indices, max_new_tokens):
        # Generate only the specified number of tokens, max_new_tokens
        for _ in range(max_new_tokens):
            input_conditioned = input_indices[:, -self.config.input_sequence_length:] # Crop the input

            # The forward pass returns `(likelihood, loss)` -- keep only `likelihood` as `logits`.
            logits, _ = self.forward(input_conditioned, target_indices=None)
            last_logits = logits[:, -1, :] # Extract the logits for the last token
            probs = F.softmax(last_logits, dim=-1) # Convert likelihood to probabilities with Softmax

            # Sample the next token
            next_token = torch.multinomial(probs, num_samples=1)

            # Add the new token and update input_indices.
            input_indices = torch.cat((input_indices, next_token), dim=1)

        # Return the final `input_indices`. Its length is the original `input_indices` + `max_new_tokens`
        return input_indices

    # Calculate likelihood and loss
    def forward(self, input_indices, target_indices):
        embeddings = self.embedding(input_indices)
        blocks_output = self.blocks(embeddings)
        logits = self.vocab_projection(blocks_output)

        # During inference, there is no target, so loss is None
        # -- only the probabilities (logits) are returned.
        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits = logits.view(batch_size * token_len, vocab_size)
        targets = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits, targets)

        return logits, loss


----

**Heads up!** Let’s train for 10,000 steps with batch size 16.




In [ ]:
# A config class that stores the model settings
class ModelConfig:
    batch_size = 16
    input_sequence_length = 512  # Use a longer sequence at once to reduce the share of time spent on data loading.
    ########## NEW ##########
    total_training_steps = 10_000  # Train thoroughly using the GPU's computing power
    ########## NEW ##########
    device_type = 'cuda'  # Fix the device to GPU
    evaluation_frequency = 100  # How often to evaluate model performance
    learning_rate = 0.001  # Learning rate
    evaluation_loops = 10  # Number of loops during evaluation
    embedding_dim = 64  # Embedding layer size (number of dimensions in the feature vector)
    hidden_dim = 256
    num_attention_heads = 4  # Number of attention heads
    layer_count = 4  # Number of model layers
    dropout_rate = 0.1  # Dropout probability
    random_seed_value = 1337  # Random seed for reproducibility

In [ ]:
# Load the settings and set the seed
config = ModelConfig()
torch.manual_seed(config.random_seed_value)  # Set the random seed for reproducibility

In [ ]:
# Load the data
with open("input.txt", 'r', encoding = 'utf-8') as f:
    text_data = f.read()
data_loader = DataLoader(text_data, config)

In [ ]:
# Initialize the model and optimizer
model = nanoGPT(vocab_size = data_loader.vocab_size, config = config).to(config.device_type)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

In [ ]:
# Display the number of model parameters
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Training takes about 10 minutes. While it runs, work on the matplotlib fill-in-the-blanks question in the next cell!




In [ ]:
print("===Training started successfully===")

# Train the model
trainer = Trainer(model, optimizer, data_loader, config)
trainer.train()

Let’s use `matplotlib` to draw a graph with `Step` on the x-axis and `Loss` on the y-axis.


🔘 **Options**: There may be extra choices. You may use the same choice more than once.

`trainer.train_losses`　`trainer.val_losses`　`trainer.steps`　`'Step'`　`'Loss'`　`plt.legend()`　`plt.show()`


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')
plt.plot(_____________, _______________, label='Validation Loss')
plt.xlabel(_____)
plt.ylabel(_____)
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
________

<details>
<summary>Click to show/hide the answer</summary>

```python
# Draw the graph.
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')
plt.plot(trainer.steps, trainer.val_losses, label='Validation Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()
```


**Training Progress and loss plateau**

1. **Early stage**: Both training error and validation error drop quickly.
2. **Middle stage**: Both errors decrease more slowly.
3. **Late stage**: Validation error becomes almost flat, while only training error keeps going down a little.

---

A state where **validation error barely improves anymore** is called a **loss plateau**.

A plateau is a high, flat area of land. It is a neat and elegant name for this pattern.

The model has reached the limit of its expressive power, so adding more compute will not improve it much.

Instead, compute resources should mainly be invested in these two things:

1. **Make the model larger** (increase layers and parameters)
2. **Increase the data** (train on more diverse samples)

---

In later Chapters, these two topics, **model scaling** and **data expansion**, will become the main themes.


In [ ]:
# Switch to evaluation mode. Disable dropout.
model.eval()
print("Model set to eval mode")

In [ ]:
prompt = "Let's he"
print(f"\nInput prompt: {prompt}")

# Tokenize and convert to a tensor
encoded = data_loader.encode(prompt) # Encode the text into IDs
print("encoded", encoded)
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # Convert the list of IDs into tensor format
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.unsqueeze(0)  # Add the batch dimension
print_formatted_tensor(encoded_tensor)
encoded_tensor = encoded_tensor.to(config.device_type) # Transfer encoded_tensor to cuda (GPU)
print_formatted_tensor(encoded_tensor)

In [ ]:
# Generate text
generated_text = model.generate(encoded_tensor, max_new_tokens=512)

In [ ]:
decoded_text = data_loader.decode(generated_text[0].tolist())
print(decoded_text)

Let’s compare this with the result from 5 minutes of CPU training in Chapter12.


```plain
Let's here.

CLOUCESTIO:
She beep shall in mips a Preoped:
'Tis now distake my sore, and fould, the, for morture,
And for boar here, right I can derancious!

PRULIET:
Pomcticp face are high'd laut rovicts becumse,
The let might's must happre my
Lor my coury drease.

GLOUCESTER:
I sincand they mmudes where orweet;
Applentle, you at let of the conforth likes,
Beforeing, and God word, not fieve:
I than not tublf arust stre
For must rawes! my ever true.

LOUCESTER:
Care wass fouly to not me condlegenceit, have the with
```

Since this is old-style English in the first place, it may not really click, right? lol 😛

If you trained on the GPU for 10 minutes, you might feel like the spelling is just a little more correct...? That is about the difference.

The model size is also very small at 0.26M. The dataset is also very small, about 100,000 characters.

Let’s make both bigger and improve performance.

We still have two more Chapters with old-style English Shakespeare. Please hang in there just a little longer!

We will move to a modern dataset too!




**Section 2: loss plateau** <label><input type="checkbox"> Mark as Done</label>

**⚠️ Please disconnect the runtime from the 🔽 in the top-right corner to stop using credits.** <label><input type="checkbox">Disconnected</label>


**`Chapter 15: Train nanoGPT with GPU`** <label><input type="checkbox"> Mark as Done</label>